In [4]:
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient, models

In [38]:
embedder_url = 'http://127.0.0.1:8001/v1'
qdrant_url = 'http://127.0.0.1:6333'

embedder = OpenAI(
    base_url=embedder_url,
    api_key='no-sk'
)
qclient = QdrantClient(
    url=qdrant_url,
    api_key='1234'
)

/tmp/ipykernel_2016579/649364465.py:8: UserWarning: Api key is used with an insecure connection.
  qclient = QdrantClient(


In [ ]:
#TFIDF Temrn Frequency Inverse Document Frequency 
# toutes les mots qui ne sont pas une bon discriminante sont met a 0
#rechereche par mot cles similaire par mot cles
# accetnt, cntre sur le racine , 1. revien suvant dan corpus and 2. suvan dansl le chunk, 
# 3 est ill est present souvant ou non, si pas souvant il est pas une bon discriminatuer dans different documents 
#"Est-ce que le vendeur Bob Johny est le plus rentable"
#[0.0,]

[0.0]

In [17]:
text = "Les dividendes pour l'annee 2026 seront de 10M"
#text = "Est-ce que le vendeur Bob Johny est le plus rentable"

r = embedder.embeddings.create(model='bge-m3', input=text)
r.data[0].embedding
vector = np.array(r.data[0].embedding)
vector

array([-0.05665321, -0.01017404, -0.00757681, ...,  0.00684809,
       -0.02490723,  0.00803926], shape=(1024,))

In [18]:
import uuid
file_path = 'export_data/exemple.pdf'
chunk_id = 1

#from uuid import uuid5
id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{chunk_id} - {file_path}'))

id


'65113525-6fa1-52f1-9b4c-fc7045515139'

In [19]:
#langue doive etre detecte an avance 
# a cause que dans different langue le division du mot ce fait differente
options = {
'language': 'french',
'ascii_folding': True,
'avg_len': 50
}

sparse_vector = models.Document(text=text, model='qdrant/bm25', options=options)
sparse_vector

Document(text="Les dividendes pour l'annee 2026 seront de 10M", model='qdrant/bm25', options={'language': 'french', 'ascii_folding': True, 'avg_len': 50})

In [44]:
payload = {
    'file_extenson':'pdf',
    'chunk_id':1,
    'file_name':'exemple.pdf',
    'extraction_date': '2026-01-01',
    'text': text
}

point = models.PointStruct(
    id=1,
    vector = {'dense': vector, 'sparse': sparse_vector},
    payload=payload

)

In [ ]:
collection = 'demo_collection'

if not qclient.collection_exists(collection):
    qclient.create_collection(
        collection,
        vectors_config={'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE)},      
        sparse_vectors_config={'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)}
    )

    qclient.create_payload_index(collection, field_name= 'file_path', field_schema=models.PayloadSchemaType.KEYWORD)

    #sparse name should be the same as above to work

In [46]:
qclient.upsert(collection, points=[point,])

#possible de inserte en block pour etre plus eficace

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)